# International Football Match Predictor

Completed notebook for the IBM SkillsBuild AI Builders Challenge June 2026 football lab. This notebook follows the official lab flow and uses reusable Python modules from this repository so the prototype can run both in Jupyter and as a Streamlit app.

## IBM Bob prompts used

- Help me set up a Python/Jupyter environment for the IBM SkillsBuild football prediction lab.
- Generate Python code to load and explore the international football results dataset.
- Explain how to engineer leak-resistant team features from historical match results.
- Debug model training errors and package import issues.
- Help me create a Streamlit prototype that predicts match winner probabilities and explains the result.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "data" / "results.csv").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
print(project_root)

## Load the dataset

In [ ]:
from football_predictor import load_matches

matches = load_matches(project_root / "data" / "results.csv")
print(matches.shape)
print(matches["date"].min(), matches["date"].max())
matches.head()

## Explore tournaments and teams

In [ ]:
import pandas as pd

print("Top tournaments")
print(matches["tournament"].value_counts().head(10).to_string())

all_teams = pd.concat([matches["home_team"], matches["away_team"]])
print("\nTop teams by match count")
print(all_teams.value_counts().head(15).to_string())

## Build chronological features

In [ ]:
from football_predictor import build_training_frame

features = build_training_frame(matches)
print(features.shape)
features.head()

## Train and evaluate the model

In [ ]:
from football_predictor.modeling import train_model

model, report = train_model(features)
print(f"Accuracy: {report['accuracy']:.2%}")
print(f"Baseline: {report['baseline_accuracy']:.2%}")
print(report["confusion_matrix"])
sorted(report["feature_importance"].items(), key=lambda item: item[1], reverse=True)

## Save model artifacts for the prototype

In [ ]:
from football_predictor import build_team_stats
from football_predictor.modeling import save_artifacts

team_stats = build_team_stats(matches)
save_artifacts(model, team_stats, report, project_root / "models")
print(f"Saved artifacts for {len(team_stats)} teams.")

## Try a prediction

In [ ]:
from football_predictor import predict_match
from football_predictor.modeling import explanation_lines

result = predict_match("Brazil", "Argentina", model, team_stats, is_neutral=True, is_major_tournament=True)
print(result["predicted_outcome"])
print(result["probabilities"])
for line in explanation_lines("Brazil", "Argentina", team_stats, report):
    print("-", line)

## Run the prototype

In [ ]:
print("From the project root, run:")
print("streamlit run app.py")